# Web Research Agent

Use the same internet-search tool with either an OpenAI or Gemini agent, then summarize the sources it finds.

## 1. Install dependencies

Run this cell once when starting a new Colab session.

In [ ]:
%pip install -q openai-agents requests google-genai ddgs

## 2. Choose and configure a model provider

Set `PROVIDER` to `"openai"` or `"gemini"`. Add the matching API key to Colab **Secrets** before running this cell.

In [ ]:
import os

from agents import OpenAIChatCompletionsModel, set_tracing_disabled
from google.colab import userdata
from openai import AsyncOpenAI

PROVIDER = "openai"  # Change to "gemini" to use Gemini.

if PROVIDER == "openai":
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    if not os.environ["OPENAI_API_KEY"]:
        raise ValueError("Add OPENAI_API_KEY to Colab Secrets before running this notebook.")
    model = None
elif PROVIDER == "gemini":
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
    GEMINI_MODEL_NAME = "gemini-2.5-flash"

    if not GEMINI_API_KEY:
        raise ValueError("Add GEMINI_API_KEY to Colab Secrets before running this notebook.")

    set_tracing_disabled(disabled=True)
    model = OpenAIChatCompletionsModel(
        model=GEMINI_MODEL_NAME,
        openai_client=AsyncOpenAI(
            base_url=GEMINI_BASE_URL,
            api_key=GEMINI_API_KEY,
        ),
    )
else:
    raise ValueError("PROVIDER must be either 'openai' or 'gemini'.")

## 3. Define the search tool and research agent

The tool uses the key-free `ddgs` library to return search-result titles, summaries, and links. The agent uses those results to produce a grounded summary.

In [ ]:
from ddgs import DDGS
from ddgs.exceptions import DDGSException

from agents import Agent, Runner, function_tool


@function_tool
def search_web(query: str) -> str:
    """Search the web and return relevant source titles, summaries, and URLs."""
    try:
        results = DDGS(timeout=15).text(query, max_results=5)
    except DDGSException as error:
        return f"Search is temporarily unavailable: {error}"

    if not results:
        return "No results were found. Tell the user that the search did not return enough information."

    return "\n\n".join(
        f"{result.get('title', 'Untitled source')}\n"
        f"{result.get('body', 'No summary available.')}\n"
        f"{result.get('href', 'No URL available.')}"
        for result in results
    )


agent = Agent(
    name="Research Assistant",
    instructions="""
You are a careful research assistant.

- Always call search_web before answering a research question.
- Summarize only information returned by the tool.
- If the results are insufficient or conflict, say so clearly.
- End with a Sources section that lists the URLs you used.
""",
    tools=[search_web],
    **({} if model is None else {"model": model}),
)

question = "What is agentic AI, and how is it used in business?"
result = await Runner.run(starting_agent=agent, input=question)

print(result.final_output)